# 第2回 宿題（基礎）：設定を 1 つ変えて再学習し、`result.json` を 2 つ比べる

`notebooks/lora_format_tuning.ipynb` のセクション 1〜7 と同じ処理を `run_lora.py` にまとめ、**元の設定**と **`num_train_epochs` だけを 20 → 5 に変えた設定**を、それぞれ別プロセスで上から実行する。プロセスが終わるとモデルも消えるので、「ランタイム → セッションを再起動して上から実行」と同じ条件で比べられる。

| セル | 内容 | T4 での目安 |
| --- | --- | --- |
| 1〜3 | GPU 確認、ライブラリ導入（ノートブックと同じ版）、スクリプトの書き出し | 約 2 分 |
| 4 | 元の設定（20 エポック、40 ステップ）を実行 | 約 7〜8 分 |
| 5 | 変更後（5 エポック、10 ステップ）を実行 | 約 3〜4 分 |
| 6〜7 | 比較表と `answer.md` の下書きを作り、zip をダウンロード | 1 分 |

**ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んでから、**ランタイム → すべてのセルを実行**。終わったら **ランタイムを接続解除して削除** する。

別の項目を変える場合はセル 5 の引数を `--lr 5e-5` や `--lora_r 4` に替え、`--label` も替える（1 項目だけ）。

In [ ]:
# 1. GPU の確認
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU"

In [ ]:
# 2. ライブラリ（ノートブックのセクション 0 と同じ版に固定）
%pip uninstall -y -q torchao
%pip install -q "transformers==5.18.0" "peft==0.21.1" "trl==1.14.2" "accelerate==1.15.0" "datasets==5.1.0"
# cudf / pyarrow の依存警告（赤い ERROR）は無視してよい

In [ ]:
%%writefile run_lora.py
#!/usr/bin/env python
"""第2回 宿題（基礎）: CONFIG の 1 項目だけを変えて LoRA で再学習し、result.json を作る。

notebooks/lora_format_tuning.ipynb のセクション 1〜7 と同じ処理（データ生成・プロンプト・採点・学習設定）を
1 つのプロセスで実行する。1 回の実行が 1 つの設定に対応し、プロセスが終わると GPU 上のモデルも消えるので、
「ランタイム → セッションを再起動して上から実行」と同じ条件で比べられる。

ノートブックから変えた点（学習そのものは変えていない）
- 宿題で記録する「最終 loss」と「学習時間」を result.json の "train" に記録する。
- loss を毎ステップ記録する（logging_steps=1）。ノートブックの 10 ステップごとの記録だと、学習が 10 ステップしかない
  設定では最終 loss が「1〜10 ステップの平均」になり、元の設定（31〜40 ステップの平均）と比べられないため。
  logging_steps はログの間隔だけを決める設定で、勾配や学習率には影響しない。
- 学習後に valid（7 件）の loss を 1 回だけ測る（trainer.evaluate()）。学習には使わない。

使い方（Colab の T4 で）
    python run_lora.py --label baseline                          # 元の設定（20 エポック）
    python run_lora.py --label epochs5 --num_train_epochs 5      # 1 項目だけ変える
"""
import argparse
import json
import os
import re
import time
from pathlib import Path

# ---------------------------------------------------------------- 1. 設定
BASE_CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},
}
HW_KEYS = ("lora_r", "lr", "num_train_epochs")  # 宿題（基礎）で変えてよい項目

ap = argparse.ArgumentParser()
ap.add_argument("--label", required=True, help="結果フォルダの名前（例: baseline, epochs5）")
ap.add_argument("--lora_r", type=int)
ap.add_argument("--lr", type=float)
ap.add_argument("--num_train_epochs", type=int)
ap.add_argument("--out_root", default="/content/qlora_lab/hw")
# 以下は動作確認用。宿題では既定値のまま使う
ap.add_argument("--base_model", default=BASE_CONFIG["base_model"])
ap.add_argument("--revision", default=BASE_CONFIG["revision"], help="'none' で revision を指定しない")
ap.add_argument("--max_new_tokens", type=int, default=BASE_CONFIG["max_new_tokens"])
args = ap.parse_args()

overrides = {k: getattr(args, k) for k in HW_KEYS if getattr(args, k) is not None}
assert len(overrides) <= 1, f"宿題（基礎）は 1 項目だけ変える: {overrides}"
CONFIG = dict(BASE_CONFIG, **overrides)
CONFIG["base_model"] = args.base_model
CONFIG["revision"] = None if args.revision.lower() == "none" else args.revision
CONFIG["max_new_tokens"] = args.max_new_tokens
# 注: lora_r を変えても lora_alpha は 32 のまま（1 項目だけ変える）。adapter 出力の倍率 alpha/r も同時に変わる

WORK = Path(args.out_root) / args.label
WORK.mkdir(parents=True, exist_ok=True)
print("label:", args.label, "| 変えた設定:", overrides or "なし（元の設定）")
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

# ---------------------------------------------------------------- 2. 架空 FAQ と教師データ（ノートブックと同一）
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"


def make_example(question, answer, needs_review):
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}


def split_templates(templates):
    return templates[:2], templates[2:3], templates[3:4]


train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions)
assert (len(train), len(valid), len(test), len(gold)) == (14, 7, 21, 21)
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")

# ---------------------------------------------------------------- 3. モデルの読み込みと学習前評価
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"


def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def generate(model, example):
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def check_format(text):
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj


def check_content(obj, gold_entry):
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])


def mark(ok):
    return "○" if ok else "×"


def evaluate(model, label):
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows) / n,
               "content_rate": sum(r["content_ok"] for r in rows) / n,
               "both_rate": sum(r["both_ok"] for r in rows) / n,
               "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    for r in rows:
        print(f"  [{mark(r['format_ok'])}/{mark(r['content_ok'])}] gold_nr={r['gold_needs_review']!s:5} {r['question']} -> {r['raw'][:90]}")
    return summary, rows


print("\n=== 学習前評価 ===")
before_summary, before_rows = evaluate(model, "before")

# ---------------------------------------------------------------- 4. LoRA 学習
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


def to_chat(example):
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}


train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=1,  # ノートブックは 10。ログの間隔だけの違いで学習には影響しない（冒頭の説明を参照）
                save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"],
                # GPU が無いときの動作確認用。CUDA（T4）ではノートブックと同じく何も渡さない
                **({"use_cpu": True, "bf16": False} if DEVICE == "cpu" else {}))
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"\ntrainable params: {trainable:,} || all params: {total:,} || trainable%: {100 * trainable / total:.4f}")
if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_out = trainer.train()
train_seconds = time.time() - t0
print(f"学習時間: {train_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")

loss_log = [{"step": h["step"], "loss": h["loss"], "learning_rate": h.get("learning_rate")}
            for h in trainer.state.log_history if "loss" in h]
valid_loss = trainer.evaluate().get("eval_loss")   # 学習後の valid loss（学習には使わない）


def window_mean(lo, hi):
    xs = [e["loss"] for e in loss_log if lo <= e["step"] <= hi]
    return sum(xs) / len(xs) if xs else None


steps = trainer.state.global_step
train_info = {
    "global_step": steps,
    "epoch": trainer.state.epoch,
    "train_seconds": round(train_seconds, 1),                   # ノートブックの「学習時間」と同じ測り方（time.time() の差）
    "train_runtime": round(train_out.metrics.get("train_runtime", float("nan")), 1),
    "final_loss": loss_log[-1]["loss"] if loss_log else None,  # 最終ステップの loss
    "final_loss_last10_mean": window_mean(steps - 9, steps),    # 最後の 10 ステップの平均（ノートブックの表示に相当）
    "mean_train_loss": train_out.training_loss,                 # 全ステップの平均
    "valid_loss": valid_loss,
    "trainable_params": trainable, "all_params": total,
    "trainable_pct": 100 * trainable / total,
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE == "cuda" else None,
    "loss_log": loss_log,
}
print(json.dumps({k: v for k, v in train_info.items() if k != "loss_log"}, ensure_ascii=False))
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

# ---------------------------------------------------------------- 5. 学習後評価
print("\n=== 学習後評価 ===")
after_summary, after_rows = evaluate(tuned, "after")
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# ---------------------------------------------------------------- 6. 知識を問う質問
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",
    "担当教員の名前は何ですか",
    "次回の講義のテーマは何ですか",
    "課題の提出期限はいつですか",
]


def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"


knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    # 注: ノートブックと同じく "FAQ にある" は q in gold で判定する。対照の「課題の提出期限はいつですか」は
    # train にある質問で gold（test）に無いため False と表示される（ノートブック側の表示上の問題で、採点には影響しない）
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                           "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
    print(f"  {q}: {classify(before_raw)} -> {classify(after_raw)} | {after_raw[:80]}")

# ---------------------------------------------------------------- 7. result.json
import transformers, peft, trl
result = {
    "label": args.label,
    "changed": overrides,
    "config": CONFIG,
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE,
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "train": train_info,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n保存しました:", WORK / "result.json")

In [ ]:
%%writefile compare.py
#!/usr/bin/env python
"""第2回 宿題（基礎）: run_lora.py の result.json を 2 つ並べ、比較表と answer.md の下書きを作る。

    python compare.py --baseline /content/qlora_lab/hw/baseline --variant /content/qlora_lab/hw/epochs5 --out /content/qlora_lab/hw

出力
- comparison.md : 比較表と、学習後に落ちた質問の一覧（失敗の型つき）
- answer.md     : lectures/02/answer.md にそのまま貼れる下書き（演習欄は元の設定の実行結果から作る）
考察の文は数字に応じて組み立てているが、最後は出力原文を自分で読んで直すこと。
"""
import argparse
import json
from pathlib import Path

FAQ_SENTENCES = {
    "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "提出先は学習ポータルの「課題」欄です。",
    "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
    "この FAQ には記載がありません。運営に確認が必要です。",
}
CONTROL_Q = "課題の提出期限はいつですか"   # 知識質問のうち FAQ にある対照（ノートブックの "FAQ にある" 列は False と出るが、FAQ にある）
KEY_JA = {"num_train_epochs": "num_train_epochs（エポック数）", "lr": "lr（学習率）", "lora_r": "lora_r（LoRA の次元）"}

ap = argparse.ArgumentParser()
ap.add_argument("--baseline", required=True)
ap.add_argument("--variant", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()


def load(d):
    d = Path(d)
    res = json.loads((d / "result.json").read_text(encoding="utf-8"))
    after = json.loads((d / "eval_after.json").read_text(encoding="utf-8"))["rows"]
    before = json.loads((d / "eval_before.json").read_text(encoding="utf-8"))["rows"]
    return res, before, after


B, B_before, B_after = load(args.baseline)
V, V_before, V_after = load(args.variant)
assert B["changed"] == {}, "baseline は元の設定で実行したものを指定する"
assert len(V["changed"]) == 1, "variant は 1 項目だけ変えたものを指定する"
KEY, VAL = next(iter(V["changed"].items()))
BVAL = B["config"][KEY]
N = B["after"]["n"]


def rate(x, n=N):
    return f"{x:.3f}（{round(x * n)}/{n}）"


def f4(x):
    return "—" if x is None else f"{x:.4f}"


def failure_type(r):
    if r["both_ok"]: return None
    if not r["format_ok"]: return f"形式の崩れ（{r['format_reason']}）"
    obj = json.loads(r["raw"])
    if r["gold_needs_review"] and obj["needs_review"] is False: return "根拠なしの質問に断定"
    if not r["gold_needs_review"] and obj["needs_review"] is True: return "根拠ありの質問に確認要求で逃げた"
    return "needs_review は正しいが必須語句が不足"


def answer_of(r):
    try: return json.loads(r["raw"]).get("answer")
    except Exception: return None


def failures(rows):
    return [(r["question"], failure_type(r), r["raw"]) for r in rows if not r["both_ok"]]


def verbatim_share(rows):
    ok = [r for r in rows if r["format_ok"]]
    return sum(answer_of(r) in FAQ_SENTENCES for r in ok), len(ok)


def knowledge_line(res):
    return " / ".join(f"「{k['質問']}」{k['学習前の分類']} → {k['学習後の分類']}" for k in res["knowledge"])


def nonfaq_asserted(res):
    return [k for k in res["knowledge"] if k["質問"] != CONTROL_Q and k["学習後の分類"] == "asserted"]


def fail_md(fs):
    if not fs: return "なし"
    return "<br>".join(f"「{q}」: {t} — `{raw}`" for q, t, raw in fs)


tb, tv = B["train"], V["train"]

# ------------------------------------------------------------ 比較表
rows = [
    ("変えた設定", f"{KEY} = {BVAL}", f"{KEY} = {VAL}"),
    ("学習ステップ数 / エポック", f"{tb['global_step']} / {tb['epoch']:.1f}", f"{tv['global_step']} / {tv['epoch']:.1f}"),
    ("学習前 format_rate / content_rate", f"{rate(B['before']['format_rate'])} / {rate(B['before']['content_rate'])}",
     f"{rate(V['before']['format_rate'])} / {rate(V['before']['content_rate'])}"),
    ("**学習後 format_rate**", rate(B["after"]["format_rate"]), rate(V["after"]["format_rate"])),
    ("**学習後 content_rate**", rate(B["after"]["content_rate"]), rate(V["after"]["content_rate"])),
    ("学習後 both_rate", rate(B["after"]["both_rate"]), rate(V["after"]["both_rate"])),
    ("退行した質問", str(len(B["regressed"])), str(len(V["regressed"]))),
    ("**最終 loss**（最終ステップ）", f4(tb["final_loss"]), f4(tv["final_loss"])),
    ("最後の 10 ステップの loss 平均", f4(tb["final_loss_last10_mean"]), f4(tv["final_loss_last10_mean"])),
    ("全ステップの loss 平均", f4(tb["mean_train_loss"]), f4(tv["mean_train_loss"])),
    ("学習後の valid loss（7 件）", f4(tb["valid_loss"]), f4(tv["valid_loss"])),
    ("**学習時間（秒）**", f"{tb['train_seconds']:.1f}", f"{tv['train_seconds']:.1f}"),
    ("trainable%", f"{tb['trainable_pct']:.3f}", f"{tv['trainable_pct']:.3f}"),
    ("知識質問（学習前 → 学習後）", knowledge_line(B), knowledge_line(V)),
]
table = "| 指標 | 元の設定（baseline） | 変更後（" + V["label"] + "） |\n| --- | --- | --- |\n" + \
        "\n".join(f"| {a} | {b} | {c} |" for a, b, c in rows)

diff_q = []
for b, v in zip(B_after, V_after):
    assert b["question"] == v["question"]
    if b["both_ok"] != v["both_ok"]:
        diff_q.append(f"- 「{b['question']}」: baseline {'○' if b['both_ok'] else '×（' + failure_type(b) + '）'} / "
                      f"{V['label']} {'○' if v['both_ok'] else '×（' + failure_type(v) + '）'}")
diff_md = "\n".join(diff_q) if diff_q else "- なし（どちらの設定でも同じ質問が○／×になった）"

comparison = f"""# 宿題（基礎）比較: {KEY} {BVAL} → {VAL}

{table}

## 学習後に落ちた質問（失敗の型）

- baseline: {fail_md(failures(B_after))}
- {V['label']}: {fail_md(failures(V_after))}

## 2 つの設定で結果が分かれた質問

{diff_md}

## loss の推移（step: loss）

- baseline: {", ".join(f"{e['step']}:{e['loss']:.3f}" for e in tb["loss_log"])}
- {V['label']}: {", ".join(f"{e['step']}:{e['loss']:.3f}" for e in tv["loss_log"])}

実行環境: {B.get('device')} / {json.dumps(B['versions'], ensure_ascii=False)}
"""

# ------------------------------------------------------------ 考察（数字に応じて組み立てる）
fb, fv = B["after"]["format_rate"], V["after"]["format_rate"]
cb, cv = B["after"]["content_rate"], V["after"]["content_rate"]
sb, sv = tb["global_step"], tv["global_step"]
desc = f"{KEY} を {BVAL} から {VAL} に変える（学習ステップ {sb} → {sv}）と"
s = []
if abs(fv - fb) < 1e-9:
    s.append(f"{desc}、学習後の形式遵守率はどちらも {fv:.3f} で差は出なかった。")
    if sv < sb and fv == 1.0:
        s.append(f"JSON の形式は {sv} ステップの学習でも完全に身についた。")
else:
    s.append(f"{desc}、学習後の形式遵守率は {fb:.3f} → {fv:.3f} に{'下がった' if fv < fb else '上がった'}。")
    if fv < fb and sv < sb:
        s.append("形式の習得にも一定の学習量が要る。")
dc = round((cv - cb) * N)
if dc == 0:
    s.append(f"内容正解率もどちらも {cv:.3f} だった。")
else:
    s.append(f"内容正解率は {cb:.3f} → {cv:.3f}（{abs(dc)} 問{'減' if dc < 0 else '増'}）だった。")
    if abs(dc) <= 2:
        s.append(f"ただし 1 問は約 {100 / N:.1f} ポイントに当たり、21 問・1 回の実行（GPU 上の学習は seed を固定しても完全には再現しない）では、1〜2 問の差は誤差の範囲として扱うべきである。")
s.append(f"最終ステップの loss は {f4(tb['final_loss'])} → {f4(tv['final_loss'])}、valid loss は {f4(tb['valid_loss'])} → {f4(tv['valid_loss'])}、"
         f"学習時間は {tb['train_seconds']:.0f} 秒 → {tv['train_seconds']:.0f} 秒だった。")
if tb["final_loss"] is not None and tv["final_loss"] is not None:
    lo_name, lo_c, hi_c = ("元の設定", cb, cv) if tb["final_loss"] < tv["final_loss"] else (V["label"], cv, cb)
    if lo_c <= hi_c:
        s.append(f"train loss が低い方（{lo_name}）の内容正解率は、もう一方と{'同じ' if lo_c == hi_c else 'くらべてむしろ低い'}だった。"
                 "loss は教師文をどれだけ再現できるかの指標であり、FAQ に根拠があるかの判断の正しさまでは表さない。")
if KEY == "num_train_epochs":
    w = B["config"]["warmup_steps"]
    s.append(f"なお warmup_steps={w} はステップ数で固定されているため、{sv} ステップの学習では warmup が全体の {100 * min(w, sv) / sv:.0f}%（元の設定では {100 * w / sb:.0f}%）を占める。"
             "エポック数の変更は、更新回数だけでなく学習率スケジュールの形も変えている。")
    if sb and sv:
        s.append(f"学習時間はステップ数の比（{sv / sb:.2f}）に対して {tv['train_seconds'] / tb['train_seconds']:.2f} 倍で、ほぼステップ数に比例する。")
if KEY == "lora_r":
    s.append(f"trainable% は {tb['trainable_pct']:.3f} → {tv['trainable_pct']:.3f} と変わったが、学習時間の比は {tv['train_seconds'] / tb['train_seconds']:.2f} 倍だった。"
             "lora_alpha は 32 のまま固定したので、adapter 出力の倍率 alpha/r も同時に変わっている。")
p_compare = "".join(s)

k_share, k_n = verbatim_share(B_after)
asserted_test = [(q, raw) for q, t, raw in failures(B_after) + failures(V_after) if t == "根拠なしの質問に断定"]
asserted_know = nonfaq_asserted(B) + nonfaq_asserted(V)
fb0 = B["before"]["format_rate"]
if fb > fb0:
    p_good = (f"ファインチューニングが向くのは、出力の形式・文体・定型の振る舞いを揃えたい場面である。今回、プロンプトで JSON 形式を明示しても学習前の形式遵守率は "
              f"{fb0:.3f} だったが、全パラメータの {tb['trainable_pct']:.2f}% だけを学習する LoRA で {fb:.3f} になった。"
              "形式の要件をプロンプトに長く書かずに済み、推論 1 回あたりのコストも増えない。出力を後段のプログラムが機械的に読む場面（JSON、分類ラベル、定型の確認メッセージ）では効果が大きい。")
else:
    p_good = (f"ファインチューニングは出力形式を揃える用途で使われることが多いが、今回は学習後も形式遵守率が {fb0:.3f} → {fb:.3f} で改善せず、"
              "この用途での効果は今回の結果からは確かめられなかった。")
if asserted_test or asserted_know:
    if asserted_know:
        ex_q, ex_a = asserted_know[0]["質問"], asserted_know[0]["学習後の出力"]
    else:
        ex_q, ex_a = asserted_test[0]
    evid = (f"FAQ に無い質問に、FAQ の別項目を流用して needs_review: false で断定する出力が残った（例:「{ex_q}」→ `{ex_a}`）。"
            "質問の語が FAQ の項目と表面上似ているかで答えを選んでいるように見え、「根拠があるか」という判断そのものが身についたとは言えない（今回の出力から見た仮説で、検証はしていない）。")
else:
    evid = ("今回の test と知識質問では断定は見られなかったが、根拠なしの test は 6 問、知識質問は 3 問しかなく、判断が身についたと結論するには少なすぎる。")
p_bad = (f"向かないのは、新しい知識を覚えさせたい場面や、事実が頻繁に変わる場面である。学習後の回答は、形式を守れた出力のうち "
         f"{k_share}/{k_n} 件が教師データの文（FAQ 原文か確認要求の定型文）とまったく同じで、学習したのは主に「どの定型文を返すか」だった。{evid}"
         "また FAQ の内容（締切など）が変わるたびに再学習が必要で、答えがどの文書に基づくかも示せない。"
         "事実を答える部分は検索（RAG）で根拠を入力に渡して更新を容易にし、ファインチューニングは出力形式と「根拠が無いときは確認を求める」振る舞いの補強に使う、という分担が妥当である。")

# ------------------------------------------------------------ 演習欄（元の設定の実行から）
c = B["config"]
cfg_line = (f"`{c['base_model']}`（revision `{str(c['revision'])[:7]}`）、LoRA（量子化なし）、lora_r={c['lora_r']}、lora_alpha={c['lora_alpha']}、"
            f"lr={c['lr']}、num_train_epochs={c['num_train_epochs']}（{tb['global_step']} ステップ）、warmup_steps={c['warmup_steps']}、seed={c['seed']}、"
            f"貪欲生成（do_sample=False）、{B.get('device')}")
b_fail = failures(B_after)
b_fail_txt = "なし" if not b_fail else "；".join(f"「{q}」→ {t}（`{raw}`）" for q, t, raw in b_fail)
changed_txt = (f"形式遵守率が {B['before']['format_rate']:.3f} → {fb:.3f}、内容正解率が {B['before']['content_rate']:.3f} → {cb:.3f} に上がった。"
               f"学習後の answer は FAQ 原文をほぼそのまま返す形になった（形式を守れた出力の {k_share}/{k_n} 件が教師データの文と一致）。")
unchanged_txt = ("FAQ に根拠があるかの判断は保証されなかった。" + ("根拠なしの質問や知識質問で、FAQ の別項目を当てはめて断定する出力が残った。"
                 if (any(t == "根拠なしの質問に断定" for _, t, _ in b_fail) or nonfaq_asserted(B)) else
                 "今回は test・知識質問とも断定は出なかったが、根拠なしの問題数が少なく、判断が身についたとまでは言えない。"))

answer = f"""# 第2回 解答

## 演習

- 使ったモデル・設定（`result.json` の `config` から）：{cfg_line}
- 自分で学習を実行したか（`executed_training_myself`）：はい（実行できた範囲：セクション 0〜7 と同じ処理を Colab の T4 で実行。宿題の比較と条件を揃えるため、ノートブックと同じ処理を 1 プロセスで行う `homework/run_lora.py` で実行した）
- 学習前：`format_rate` = {rate(B['before']['format_rate'])} / `content_rate` = {rate(B['before']['content_rate'])}
- 学習後：`format_rate` = {rate(fb)} / `content_rate` = {rate(cb)}
- 退行した質問：{'、'.join(B['regressed']) if B['regressed'] else 'なし'}
- 学習後に残った失敗と、その型（根拠なしの質問に断定した / 根拠ありの質問に確認要求で逃げた / 形式の崩れ）：{b_fail_txt}
- 知識を問う質問（セクション 6）の結果（学習前の分類 → 学習後の分類）と気づいたこと：{knowledge_line(B)}。対照の「{CONTROL_Q}」は FAQ にある質問（ノートブックの「FAQ にある」列は `q in gold` で判定しているため False と表示されるが、train に含まれる質問である）。
- 学習で変わったこと・変わらなかったこと：{changed_txt}{unchanged_txt}

## 宿題

- 取り組んだレベル：基礎
- 変えた設定やデータ：`{KEY}` を {BVAL} → {VAL} に変更（学習ステップ {sb} → {sv}）。その他の設定・データ・評価は元の設定と同じ。設定ごとに別プロセスで上から実行し（セッション再起動と同じ条件）、`result.json` を 2 つ比べた。最終 loss を正確に取るため loss を毎ステップ記録した（`logging_steps=1`。ログの間隔だけの変更で学習には影響しない）。結果ファイルは `lectures/02/homework/results/` に置いた
- 結果（比較表や数字）：

{table}

  2 つの設定で結果が分かれた質問：

{diff_md}

- ファインチューニングが向く場面・向かない場面についての考察：
  - 今回の比較から：{p_compare}
  - {p_good}
  - {p_bad}
  - AI ツールの利用：比較用スクリプトと answer.md の下書きの作成に AI（Claude）を使った。数値は自分の Colab 実行の `result.json` から自動で転記し、`eval_after.json` の出力原文を読んで失敗の型を確認した。
"""

out = Path(args.out)
out.mkdir(parents=True, exist_ok=True)
(out / "comparison.md").write_text(comparison, encoding="utf-8")
(out / "answer.md").write_text(answer, encoding="utf-8")
print(comparison)
print("書き出し:", out / "comparison.md", out / "answer.md")

In [ ]:
# 4. 元の設定（num_train_epochs=20、40 ステップ）
!python run_lora.py --label baseline

In [ ]:
# 5. 1 項目だけ変えた設定（num_train_epochs=5、10 ステップ）
!python run_lora.py --label epochs5 --num_train_epochs 5

In [ ]:
# 6. 比較表と answer.md の下書き
!python compare.py --baseline /content/qlora_lab/hw/baseline --variant /content/qlora_lab/hw/epochs5 --out /content/qlora_lab/hw
from IPython.display import Markdown, display
display(Markdown(open("/content/qlora_lab/hw/answer.md", encoding="utf-8").read()))

In [ ]:
# 7. 提出用ファイルを zip にまとめてダウンロード（adapter とチェックポイントは含めない）。Drive がマウントされていればコピーも置く
import shutil, zipfile, os
from pathlib import Path
root = Path("/content/qlora_lab/hw")
zpath = Path("/content/lec02_homework_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["answer.md", "comparison.md"]:
        z.write(root / name, f"lectures/02/homework/results/{name}")
    for label in ["baseline", "epochs5"]:
        for name in ["result.json", "eval_before.json", "eval_after.json"]:
            z.write(root / label / name, f"lectures/02/homework/results/{label}/{name}")
print("作成:", zpath)
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(zpath, "/content/drive/MyDrive/lec02_homework_results.zip"); print("Drive にもコピーしました")
from google.colab import files
files.download(str(zpath))

## 提出

1. zip を展開し、`lectures/02/homework/results/` を自分の fork の `lec02` ブランチに置く。
2. `results/answer.md` の内容を確認・手直しして `lectures/02/answer.md` に貼る（出力原文を読んで、失敗の型と考察が自分の結果と合っているかを確かめる）。
3. `lec02` から自分の fork の `main` へ PR を出す。
4. **ランタイム → ランタイムを接続解除して削除**。